# Dataset Preprocessing

This notebook shows how data are loaded, pre-processed and the result of these steps. Dependencies for executing this notebook are presented in [requirements.txt](/requirements.txt) file. Reach to [README.md](/README.md) for more information about the repository dependencies and how to install them.

In [ ]:
import sys
sys.path.append("..")

from src.experiments.data_utils import load_all_datasets
from transformers import AutoTokenizer

`load_all_datasets` function loads LeNERBr and ASSIN2 datasets from huggingface or use the cache in case it is already downloaded.

In [ ]:
ner_ds, sts_ds, rte_ds, label_list = load_all_datasets()

## STS and RTE

These 2 downstream tasks are encompassed in the ASSIN2 dataset.

It is possible to have an overview of the features and length of the subsets that compose the dataset just printing them.

In [ ]:
print(sts_ds, rte_ds)

In the dataset it is possible to see that a pair of sentences are annotated with a `relatedness_score` (average score between annotators) and a `entailment_judgement` (only pairs that had major agreement between four annotators). `relatedness_score` refers to the quantitative degree of similarity between the `premise` and `hypothesis` (1 to 5) whereas the `entailment_judgement` classifies whether a premise implies into a hypothesis, classifying into _entailment_ (1) and _non-entailment_ (0)

In [ ]:
sts_train = sts_ds["train"].to_pandas()
sts_train.head()

It is also possible to see that the statistics of the labels and the balancement of entailment classes.

In [ ]:
print("relatedness_score", f"mean: {sts_train['relatedness_score'].mean()}", f"std: {sts_train['relatedness_score'].std()}")
sts_train["entailment_judgment"].value_counts()


## NER

This task is a part of LeNERBr dataset, a dataset for Named Entity Recognition on the juridical field.

In [ ]:
ner_ds

In [ ]:
ner_train = ner_ds["train"].to_pandas()
ner_train.head()

This dataset needs to be pre-processed as the tokenization process may divide words into sub-word tokens and so labels need to be attached to each of the sub-word tokens created from the original word.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("../models/mmbert_portuguese_trimmed_unlimited")
def preprocess_ner(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True, 
        return_overflowing_tokens=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_ner = ner_ds.map(lambda x: preprocess_ner(x, tokenizer), batched=True)

In [ ]:
tokenized_ner_train = tokenized_ner["train"].to_pandas()
tokenized_ner_train.head()


In [ ]:
label_list